In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

import warnings

warnings.filterwarnings("ignore")

# preparaing data
df = pd.read_csv('./../data/ML-EdgeIIoT-dataset.csv')
df.head()

,frame.time,ip.src_host,ip.dst_host,arp.dst.proto_ipv4,arp.opcode,arp.hw.size,arp.src.proto_ipv4,icmp.checksum,icmp.seq_le,icmp.transmit_timestamp,...,mqtt.proto_len,mqtt.protoname,mqtt.topic,mqtt.topic_len,mqtt.ver,mbtcp.len,mbtcp.trans_id,mbtcp.unit_id,Attack_label,Attack_type
0,6.0,192.168.0.152,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM
1,6.0,192.168.0.101,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM
2,6.0,192.168.0.152,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM
3,6.0,192.168.0.101,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM
4,6.0,192.168.0.152,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM


In [2]:
numerical_col = []
categorical_col = []

for i in df.columns:
    if df[i].nunique() == 1: 
        df.drop(i , axis =1 , inplace = True)
        print("dropped column : ", i)
    
    elif i != 'Attack_label' or i != 'Attack_type':
        if df[i].dtype == object or df[i].nunique() <= 20:
            categorical_col.append(i)
        else:
            numerical_col.append(i)

traget_col = ['Attack_label', 'Attack_type']
print(f"Colunas target: {traget_col}") 
print(f"Colunas categóricas: {categorical_col}")
print(f"Colunas numéricas {numerical_col}")

dropped column :  icmp.unused
dropped column :  http.tls_port
dropped column :  dns.qry.type
dropped column :  dns.retransmit_request_in
dropped column :  mqtt.msg_decoded_as
dropped column :  mbtcp.len
dropped column :  mbtcp.trans_id
dropped column :  mbtcp.unit_id
Colunas target: ['Attack_label', 'Attack_type']
Colunas categóricas: ['frame.time', 'ip.src_host', 'ip.dst_host', 'arp.dst.proto_ipv4', 'arp.opcode', 'arp.hw.size', 'arp.src.proto_ipv4', 'http.file_data', 'http.request.uri.query', 'http.request.method', 'http.referer', 'http.request.full_uri', 'http.request.version', 'http.response', 'tcp.connection.fin', 'tcp.connection.rst', 'tcp.connection.syn', 'tcp.connection.synack', 'tcp.flags', 'tcp.flags.ack', 'tcp.options', 'tcp.payload', 'tcp.srcport', 'dns.qry.name.len', 'dns.retransmission', 'dns.retransmit_request', 'mqtt.conack.flags', 'mqtt.conflag.cleansess', 'mqtt.conflags', 'mqtt.hdrflags', 'mqtt.len', 'mqtt.msg', 'mqtt.msgtype', 'mqtt.proto_len', 'mqtt.protoname', 'mqtt

In [3]:
zero_percent_columns = []

for col in df.columns:
    try:
        zero_ratio = (df[col] == 0).sum() / len(df)
        if zero_ratio >= 0.50:
            zero_percent_columns.append((col, zero_ratio))
            df.drop(col , axis=1 , inplace=True)
    except:
        # Skip non-numeric or problematic columns
        continue

# Display the results
for col, ratio in zero_percent_columns:
    print(f"{col}: {ratio:.2%} zeros")

print("number of dropped columns: " ,len(zero_percent_columns))
print("New DF Shape: ", df.shape)

arp.opcode: 99.00% zeros
arp.hw.size: 99.00% zeros
icmp.checksum: 90.70% zeros
icmp.seq_le: 90.08% zeros
icmp.transmit_timestamp: 99.95% zeros
http.content_length: 94.01% zeros
http.request.uri.query: 58.47% zeros
http.referer: 68.85% zeros
http.response: 95.43% zeros
tcp.connection.fin: 94.19% zeros
tcp.connection.rst: 90.59% zeros
tcp.connection.syn: 87.21% zeros
tcp.connection.synack: 97.01% zeros
tcp.len: 73.19% zeros
udp.port: 99.74% zeros
udp.stream: 90.79% zeros
udp.time_delta: 99.74% zeros
dns.qry.name: 98.95% zeros
dns.qry.name.len: 89.47% zeros
dns.qry.qu: 99.84% zeros
dns.retransmission: 99.98% zeros
dns.retransmit_request: 100.00% zeros
mqtt.conack.flags: 79.23% zeros
mqtt.conflag.cleansess: 99.21% zeros
mqtt.conflags: 99.21% zeros
mqtt.hdrflags: 96.79% zeros
mqtt.len: 97.60% zeros
mqtt.msg: 79.23% zeros
mqtt.msgtype: 96.79% zeros
mqtt.proto_len: 99.21% zeros
mqtt.protoname: 79.23% zeros
mqtt.topic: 79.23% zeros
mqtt.topic_len: 99.21% zeros
mqtt.ver: 99.21% zeros
number of 

In [4]:
y_type = df.pop('Attack_type')

In [5]:
df[["ip.src_host", "ip.dst_host"]].value_counts()

ip.src_host    ip.dst_host  
192.168.0.170  192.168.0.128    47686
192.168.0.128  192.168.0.170    42290
               0                13096
               192.168.0.101    10097
192.168.0.101  192.168.0.128     9970
                                ...  
0.13.235.151   192.168.0.128        1
0.13.30.160    192.168.0.128        1
0.13.56.166    192.168.0.128        1
0.131.68.192   192.168.0.128        1
0.106.130.148  192.168.0.128        1
Name: count, Length: 25775, dtype: int64

In [6]:
drop_columns = [
    "frame.time",            # Date and time (Alta cardialidade)
    "ip.src_host",           # Source Host (Alta cardialidade)
    "ip.dst_host",           # Destination Host (Alta cardialidade)
    "arp.dst.proto_ipv4",    # Target IP address (Alta cardialidade)
    "arp.src.proto_ipv4",    # Sender IP address (Alta cardialidade)
    "http.file_data",        # File Data (raw content, unstructured string/bytes)
    "http.request.method",   # HTTP method (likely has little variation; use only if modeling request types)
    "http.request.full_uri", # Full request URI (very high cardinality string, likely unique per request)
    "http.request.version",  # HTTP version (may have little variance and minimal predictive value)
    "tcp.dstport",           # Destination Port (label type, likely high cardinality or redundant with `tcp.srcport`)
    "tcp.options",           # TCP Options (Alta cardialidade - sequence of bytes, not directly useful without custom parsing)
    "tcp.payload",           # TCP Payload (Alta cardialidade - Dado não estruturado)
    "tcp.srcport"            # Source Port (unsigned int, possibly high cardinality, often randomized)
]

#df.drop(drop_columns, axis=1, inplace=True)
df.shape

(157800, 20)

In [7]:
objList = df.select_dtypes(include = "object").columns
objList

Index(['frame.time', 'ip.src_host', 'ip.dst_host', 'arp.dst.proto_ipv4',
       'arp.src.proto_ipv4', 'http.file_data', 'http.request.method',
       'http.request.full_uri', 'http.request.version', 'tcp.options',
       'tcp.payload', 'tcp.srcport'],
      dtype='object')

In [27]:
y_type_without_normal = y_type[y_type != "Normal"]
df = df[y_type != "Normal"]
y_type_without_normal.unique()

array(['MITM', 'Fingerprinting', 'Ransomware', 'Uploading',
       'SQL_injection', 'DDoS_HTTP', 'DDoS_TCP', 'Password',
       'Port_Scanning', 'Vulnerability_scanner', 'Backdoor', 'XSS',
       'DDoS_UDP', 'DDoS_ICMP'], dtype=object)

In [28]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()

y_type_enc = encoder.fit_transform(y_type_without_normal.astype(str))

for obj in objList:
    df[obj] = encoder.fit_transform(df[obj].astype(str))

In [29]:
df_cleaned = df.drop(objList, axis=1)
df_cleaned.dtypes

tcp.ack          float64
tcp.ack_raw      float64
tcp.checksum     float64
tcp.dstport      float64
tcp.flags        float64
tcp.flags.ack    float64
tcp.seq          float64
Attack_label       int64
dtype: object

In [30]:
df_cleaned.head()

,tcp.ack,tcp.ack_raw,tcp.checksum,tcp.dstport,tcp.flags,tcp.flags.ack,tcp.seq,Attack_label
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1


In [31]:
y_type_without_normal

0              MITM
1              MITM
2              MITM
3              MITM
4              MITM
            ...    
157795    DDoS_ICMP
157796    DDoS_ICMP
157797    DDoS_ICMP
157798    DDoS_ICMP
157799    DDoS_ICMP
Name: Attack_type, Length: 133499, dtype: object

In [32]:
print(np.unique(y_type_without_normal))
print(np.unique(y_type_enc))

['Backdoor' 'DDoS_HTTP' 'DDoS_ICMP' 'DDoS_TCP' 'DDoS_UDP' 'Fingerprinting'
 'MITM' 'Password' 'Port_Scanning' 'Ransomware' 'SQL_injection'
 'Uploading' 'Vulnerability_scanner' 'XSS']
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13]


In [33]:
y_type_without_normal.value_counts()

Attack_type
DDoS_UDP                 14498
DDoS_ICMP                14090
Ransomware               10925
DDoS_HTTP                10561
SQL_injection            10311
Uploading                10269
DDoS_TCP                 10247
Backdoor                 10195
Vulnerability_scanner    10076
Port_Scanning            10071
XSS                      10052
Password                  9989
MITM                      1214
Fingerprinting            1001
Name: count, dtype: int64

In [34]:
X, X_test, y, y_test = train_test_split(df_cleaned, y_type_enc, test_size=0.2, random_state=42)

In [35]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(random_state=42)
model.fit(X, y)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [36]:
from joblib import dump

dump(model, './../IA/weights/traditional/random_forest_model_mult.joblib')

['./../IA/weights/traditional/random_forest_model_mult.joblib']

In [37]:
results = model.predict(X_test)
print(results)

[12 12 13 ...  3 10  3]


In [40]:
target_names = list(np.unique(y_type_without_normal))
target_names

['Backdoor',
 'DDoS_HTTP',
 'DDoS_ICMP',
 'DDoS_TCP',
 'DDoS_UDP',
 'Fingerprinting',
 'MITM',
 'Password',
 'Port_Scanning',
 'Ransomware',
 'SQL_injection',
 'Uploading',
 'Vulnerability_scanner',
 'XSS']

In [41]:
print(classification_report(y_test, results, target_names=target_names))

                       precision    recall  f1-score   support

             Backdoor       0.98      0.95      0.96      1974
            DDoS_HTTP       0.74      0.72      0.73      2184
            DDoS_ICMP       0.00      0.00      0.00      2870
             DDoS_TCP       0.74      0.79      0.76      1987
             DDoS_UDP       0.43      1.00      0.60      2879
       Fingerprinting       0.96      0.22      0.36       202
                 MITM       1.00      0.26      0.41       251
             Password       0.75      0.76      0.75      1994
        Port_Scanning       0.75      0.61      0.67      2047
           Ransomware       0.97      0.87      0.91      2175
        SQL_injection       0.74      0.78      0.76      2125
            Uploading       0.84      0.80      0.82      2020
Vulnerability_scanner       0.98      0.95      0.97      1962
                  XSS       0.78      0.77      0.78      2030

             accuracy                           0.72 